# Generate Predictions for KG Triplet Evaluation

Run **any model of your choice** over the test set and write a `predict.jsonl`
in the exact schema the evaluator expects
(`KG_triplet_evaluation/scripts/evaluation.py`).

- **Input:** `test.jsonl` — the same chat-format dataset used for training:
  `{"messages": [{"role": "user", ...}, {"role": "assistant", ...}]}`.
- **Output:** `predict.jsonl` — one JSON object per line with keys
  `prompt`, `gold_raw`, `gold_parsed`, `pred_raw`, `pred_parsed`.

> This notebook **only generates predictions** — it does not score them.
> Scoring is done separately by `evaluation.py`, which reads this `predict.jsonl`.

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-deps --force-reinstall --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## 1. Configuration — choose your model here

Set `MODEL_NAME` to whatever you want to test: a Hugging Face repo, your
fine-tuned adapter (e.g. `mohar07/qwen3-0.6b-kg-triplets`), the base model
(`Qwen/Qwen3-0.6B`), or a local/Drive path. Everything else has sensible defaults.

In [ ]:
# Choose your model -----------------------------------------------------------
MODEL_NAME     = "mohar07/qwen3-0.6b-kg-triplets"  # HF repo / adapter / local path

# Model loading ---------------------------------------------------------------
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT   = True

# Generation ------------------------------------------------------------------
NO_THINK       = True   # append " /no_think" to each prompt (Qwen3) -> faster, shorter
BATCH_SIZE     = 8      # lower this if you hit out-of-memory
MAX_NEW_TOKENS = 512    # raise (e.g. 2048) if you set NO_THINK = False
TEMPERATURE    = None   # None -> greedy/deterministic (recommended for evaluation)

# Files -----------------------------------------------------------------------
TEST_PATH      = "dataset/test.jsonl"   # input; uploaded below if missing
OUTPUT_PATH    = "predict.jsonl"        # output written here
LIMIT          = None   # set an int (e.g. 50) for a quick trial; None = all rows

## 2. Load the model

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
)
FastLanguageModel.for_inference(model)        # ~2x faster generation

if tokenizer.pad_token is None:               # some models ship without a pad token
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"               # correct slicing for batched decoding

print("Loaded:", MODEL_NAME)

## 3. Get the test set

Upload `test.jsonl` when prompted (skipped automatically if it is already at
`TEST_PATH`). If you run this notebook locally instead of Colab, just set
`TEST_PATH` to your file.

In [ ]:
import os

os.makedirs(os.path.dirname(TEST_PATH) or ".", exist_ok=True)

if not os.path.exists(TEST_PATH):
    try:
        from google.colab import files
        print(f"Upload your test file — it will be saved as {TEST_PATH}")
        uploaded = files.upload()
        fname, content = next(iter(uploaded.items()))
        with open(TEST_PATH, "wb") as f:
            f.write(content)
    except ImportError:
        raise FileNotFoundError(
            f"{TEST_PATH} not found and not running in Colab. "
            "Set TEST_PATH to your local test.jsonl."
        )

print("Using test file:", TEST_PATH)

## 4. Generate predictions

For each row we feed the **user** message (`messages[0]`) to the model and keep
the **assistant** message (`messages[1]`) as gold. Records are written to
`OUTPUT_PATH` incrementally, so a disconnect will not lose finished work.

In [ ]:
import json, re
from tqdm import tqdm

def clean(text):
    """Strip Qwen3 <think> blocks and ```json code fences from raw output."""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    return text.removeprefix("```json").removeprefix("```").removesuffix("```").strip()

def parse(text):
    """Best-effort parse of a triplet JSON array -> list (or [] on failure)."""
    try:
        d = json.loads(text)
        return d if isinstance(d, list) else [d]
    except Exception:
        return []

# Load test rows
test = [json.loads(l) for l in open(TEST_PATH, encoding="utf-8") if l.strip()]
if LIMIT:
    test = test[:LIMIT]
print(len(test), "examples to predict")

suffix = " /no_think" if NO_THINK else ""
gen_kwargs = dict(max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tokenizer.eos_token_id)
gen_kwargs.update(dict(do_sample=False) if TEMPERATURE is None
                  else dict(do_sample=True, temperature=TEMPERATURE))

n_written = n_empty = 0
with open(OUTPUT_PATH, "w", encoding="utf-8") as out:
    for i in tqdm(range(0, len(test), BATCH_SIZE), desc="Generating"):
        batch = test[i : i + BATCH_SIZE]
        prompts = [
            tokenizer.apply_chat_template(
                [{"role": "user", "content": row["messages"][0]["content"] + suffix}],
                tokenize=False, add_generation_prompt=True,
            )
            for row in batch
        ]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True,
                           truncation=True, max_length=MAX_SEQ_LENGTH).to("cuda")
        outputs = model.generate(**inputs, **gen_kwargs)

        for j, row in enumerate(batch):
            new_tokens = outputs[j][inputs["input_ids"][j].shape[0]:]
            pred_raw = clean(tokenizer.decode(new_tokens, skip_special_tokens=True))
            msgs = row["messages"]
            gold_raw = msgs[1]["content"] if len(msgs) > 1 else "[]"
            record = {
                "prompt": msgs[0]["content"],
                "gold_raw": gold_raw,
                "gold_parsed": parse(clean(gold_raw)),
                "pred_raw": pred_raw,
                "pred_parsed": parse(pred_raw),
            }
            out.write(json.dumps(record, ensure_ascii=False) + "\n")
            n_written += 1
            n_empty += not record["pred_parsed"]

print(f"Wrote {n_written} records to {OUTPUT_PATH} ({n_empty} empty/unparseable predictions)")

## 5. Validate & preview

In [ ]:
import json

rows = [json.loads(l) for l in open(OUTPUT_PATH, encoding="utf-8") if l.strip()]
required = {"prompt", "gold_raw", "gold_parsed", "pred_raw", "pred_parsed"}
assert all(required <= r.keys() for r in rows), "some records are missing required keys"
print(len(rows), "records — schema OK")

r0 = rows[0]
print("\nPROMPT (first 300 chars):\n", r0["prompt"][:300], "...")
print("\nGOLD_PARSED:\n", json.dumps(r0["gold_parsed"], indent=2, ensure_ascii=False)[:600])
print("\nPRED_PARSED:\n", json.dumps(r0["pred_parsed"], indent=2, ensure_ascii=False)[:600])

## 6. Download / save `predict.jsonl`

Download the file, then drop it into the evaluator project at
`KG_triplet_evaluation/data/predict.jsonl`.

In [ ]:
# Download to your computer
try:
    from google.colab import files
    files.download(OUTPUT_PATH)
except Exception as e:
    print("Download skipped:", e)

# Optional: also copy to Google Drive
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive")
    dst = "/content/drive/MyDrive/predict.jsonl"
    shutil.copy(OUTPUT_PATH, dst)
    print("Saved to", dst)

## Next step — run the evaluator (separate project)

Place the file at `KG_triplet_evaluation/data/predict.jsonl`, then from the
`KG_triplet_evaluation/` directory run its scorer (it reads `data/predict.jsonl`
by default):

```bash
python scripts/evaluation.py
# or point at a specific file / also write the full JSON report:
python scripts/evaluation.py --input data/predict.jsonl --report eval/metrics.json
```

The evaluator reads `gold_parsed` and `pred_parsed` from each line and reports
schema, entity-F1, relation-accuracy, weight-closeness and grounding scores.